In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%bash
# RIRS_NOISES (already have this from Phase 3 setup, skip if cached)
wget -q https://www.openslr.org/resources/28/rirs_noises.zip -O rirs_noises.zip
unzip -q rirs_noises.zip -d ./

# MUSAN -- full download required, selective extraction to save disk
wget -q https://www.openslr.org/resources/17/musan.tar.gz -O musan.tar.gz
tar -xzf musan.tar.gz musan/noise musan/music

echo "Done. RIRS_NOISES/ and musan/{noise,music}/ are ready."

Done. RIRS_NOISES/ and musan/{noise,music}/ are ready.


In [3]:
DATA_ROOT = "/content/drive/MyDrive/Data_Science_Project/data"

import shutil

shutil.copy("/content/rirs_noises.zip", f"{DATA_ROOT}/rirs_noises.zip")
shutil.copy("/content/musan.tar.gz", f"{DATA_ROOT}/musan.tar.gz")


'/content/drive/MyDrive/Data_Science_Project/data/musan.tar.gz'

In [4]:
!rm musan.tar.gz
!rm rirs_noises.zip

In [3]:
import subprocess
import shutil

DATA_ROOT = "/content/drive/MyDrive/Data_Science_Project/data"

shutil.copy(f"{DATA_ROOT}/sinhala_audio.tar.gz", "/content/sinhala_audio.tar.gz")
shutil.copy(f"{DATA_ROOT}/utt_spk_text.tsv", "/content/utt_spk_text.tsv")
shutil.copy(f"{DATA_ROOT}/librispeech_clean100.tar.gz", "/content/librispeech_clean100.tar.gz")
shutil.copy(f"{DATA_ROOT}/rirs_noises.zip", "/content/rirs_noises.zip")
shutil.copy(f"{DATA_ROOT}/musan.tar.gz", "/content/musan.tar.gz")

subprocess.run(["tar", "-xzf", "/content/sinhala_audio.tar.gz", "-C", "/content/"])
subprocess.run(["tar", "-xzf", "/content/librispeech_clean100.tar.gz", "-C", "/content/"])
subprocess.run(["unzip", "-q", "/content/rirs_noises.zip", "-d", "/content/"])
subprocess.run(["tar", "-xzf", "/content/musan.tar.gz", "-C", "/content/"])

CompletedProcess(args=['tar', '-xzf', '/content/musan.tar.gz', '-C', '/content/'], returncode=0)

In [4]:
!rm /content/sinhala_audio.tar.gz
!rm /content/librispeech_clean100.tar.gz
!rm /content/rirs_noises.zip
!rm /content/musan.tar.gz

In [5]:
!pip install -q speechbrain torch torchaudio pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 51.1 MB/s eta 0:00:00


In [ ]:
"""
PHASE 3 -- Noise & reverb resilience characterization (Sinhala + English)
Uses BOTH RIRS_NOISES (SLR28) and MUSAN (SLR17) for noise diversity.
Self-contained: does not import or depend on any earlier script.

WHAT THIS DOES
--------------
Runs the SAME set of degradation conditions on BOTH a Sinhala eval set (SLR52)
and an English eval set (LibriSpeech train-clean-100), so you can directly
compare noise robustness across languages -- not just across noise corpora.

Conditions tested (per language):
    - clean (baseline)
    - reverb (RIRS_NOISES simulated RIRs)
    - additive noise from RIRS_NOISES pointsource_noises, at 15/5/0 dB
    - additive noise from MUSAN 'noise' subset, at 15/5/0 dB
    - additive music from MUSAN 'music' subset, at 10 dB
    - worst case: reverb + MUSAN noise at 5 dB

For each (language, condition) pair, reports:
    1. EER recomputed for that condition (best-case, re-tuned threshold)
    2. FAR / FRR at the threshold FIXED from that language's own clean
       baseline (realistic, deployed-system case)

Note: thresholds are calibrated PER LANGUAGE (each language has its own
clean-condition threshold), consistent with the cross-lingual score
distributions already observed in Phase 2 -- comparing against a single
shared threshold would conflate the language gap with the noise gap.

PREREQUISITES
--------------
RIRS_NOISES/simulated_rirs/**/*.wav
RIRS_NOISES/pointsource_noises/*.wav
musan/noise/**/*.wav
musan/music/**/*.wav
Sinhala audio + utt_spk_text.tsv (SLR52)
LibriSpeech/train-clean-100 (SLR12)

RUN
---
python phase3_noise_resilience.py \
    --sinhala_audio_dir ./sinhala_audio \
    --sinhala_tsv ./utt_spk_text.tsv \
    --english_dir ./LibriSpeech/train-clean-100 \
    --utts_per_speaker 4
"""

import argparse
import glob
import itertools
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio

SAMPLE_RATE = 16000

try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier


In [ ]:
# =====================================================================
# DATA DISCOVERY
# =====================================================================
def discover_sinhala_speakers(audio_dir: str, tsv_path: str) -> dict:
    metadata = pd.read_csv(tsv_path, sep="\t", header=None,
                            names=["FileID", "UserID", "Transcription"])
    found_files = glob.glob(str(Path(audio_dir) / "**" / "*"), recursive=True)
    found_files = [f for f in found_files if f.lower().endswith((".wav", ".flac"))]
    stem_to_path = {Path(f).stem: f for f in found_files}
    metadata["path"] = metadata["FileID"].map(stem_to_path)
    available = metadata.dropna(subset=["path"])
    speaker_to_files = available.groupby("UserID")["path"].apply(list).to_dict()
    print(f"[Sinhala] {len(speaker_to_files)} speakers available locally")
    return speaker_to_files


def discover_librispeech_speakers(root_dir: str) -> dict:
    speaker_to_files = {}
    for speaker_dir in sorted(Path(root_dir).iterdir()):
        if not speaker_dir.is_dir():
            continue
        files = glob.glob(str(speaker_dir / "**" / "*.flac"), recursive=True)
        if files:
            speaker_to_files[speaker_dir.name] = files
    print(f"[English] {len(speaker_to_files)} speakers available locally")
    return speaker_to_files


def build_balanced_trials(speaker_to_files: dict, utts_per_speaker: int, seed: int = 42):
    rng = random.Random(seed)
    eligible = {s: files for s, files in speaker_to_files.items() if len(files) >= 2}
    capped = {s: files[:utts_per_speaker] for s, files in eligible.items()}

    positive_pairs = []
    for spk, files in capped.items():
        positive_pairs.extend(itertools.combinations(files, 2))
    rng.shuffle(positive_pairs)

    speakers = list(capped.keys())
    file_to_speaker = {f: s for s, files in capped.items() for f in files}
    negative_pairs = []
    for f1, _ in positive_pairs:
        spk1 = file_to_speaker[f1]
        other_spk = rng.choice([s for s in speakers if s != spk1])
        other_file = rng.choice(capped[other_spk])
        negative_pairs.append((f1, other_file))

    eval_files = list({f for pair in positive_pairs for f in pair} |
                       {f for pair in negative_pairs for f in pair})
    print(f"   -> {len(capped)} speakers, {len(eval_files)} files, "
          f"{len(positive_pairs)} positive / {len(negative_pairs)} negative pairs")
    return positive_pairs, negative_pairs, eval_files


In [ ]:
# =====================================================================
# NOISE / RIR FILE DISCOVERY (two corpora, shared across languages)
# =====================================================================
def discover_rir_files(rirs_root: str) -> list:
    files = glob.glob(str(Path(rirs_root) / "simulated_rirs" / "**" / "*.wav"), recursive=True)
    if not files:
        raise RuntimeError(f"No RIR files found under {rirs_root}/simulated_rirs")
    print(f"Found {len(files)} RIR files")
    return files


def discover_rirs_noise_files(rirs_root: str) -> list:
    files = glob.glob(str(Path(rirs_root) / "pointsource_noises" / "*.wav"))
    if not files:
        raise RuntimeError(f"No noise files found under {rirs_root}/pointsource_noises")
    print(f"Found {len(files)} RIRS_NOISES point-source noise files")
    return files


def discover_musan_files(musan_root: str, subset: str) -> list:
    files = glob.glob(str(Path(musan_root) / subset / "**" / "*.wav"), recursive=True)
    if not files:
        raise RuntimeError(f"No MUSAN '{subset}' files found under {musan_root}/{subset}")
    print(f"Found {len(files)} MUSAN '{subset}' files")
    return files

In [ ]:
# =====================================================================
# AUDIO LOADING + AUGMENTATION
# =====================================================================
def load_mono_16k(path: str) -> torch.Tensor:
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    return waveform.squeeze(0)


def _rms(x: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.mean(x ** 2) + 1e-12)


def add_reverb(clean: torch.Tensor, rir_path: str) -> torch.Tensor:
    rir = load_mono_16k(rir_path)
    rir = rir / (torch.max(torch.abs(rir)) + 1e-8)
    augmented = torchaudio.functional.fftconvolve(clean, rir, mode="full")[: clean.shape[-1]]
    augmented = augmented * (_rms(clean) / (_rms(augmented) + 1e-8))
    return augmented


def add_noise_at_snr(clean: torch.Tensor, noise_path: str, snr_db: float) -> torch.Tensor:
    noise = load_mono_16k(noise_path)
    if noise.shape[-1] < clean.shape[-1]:
        reps = int(np.ceil(clean.shape[-1] / noise.shape[-1]))
        noise = noise.repeat(reps)
    noise = noise[: clean.shape[-1]]
    clean_rms = _rms(clean)
    noise_rms = _rms(noise)
    target_noise_rms = clean_rms / (10 ** (snr_db / 20))
    noise = noise * (target_noise_rms / (noise_rms + 1e-8))
    return clean + noise


def apply_condition(clean: torch.Tensor, condition: str, rir_files: list,
                     rirs_noise_files: list, musan_noise_files: list,
                     musan_music_files: list, rng: random.Random) -> torch.Tensor:
    if condition == "clean":
        return clean
    if condition == "reverb":
        return add_reverb(clean, rng.choice(rir_files))
    if condition.startswith("rirs_noise_"):
        snr = float(condition.split("_")[-1].replace("db", ""))
        return add_noise_at_snr(clean, rng.choice(rirs_noise_files), snr)
    if condition.startswith("musan_noise_"):
        snr = float(condition.split("_")[-1].replace("db", ""))
        return add_noise_at_snr(clean, rng.choice(musan_noise_files), snr)
    if condition.startswith("musan_music_"):
        snr = float(condition.split("_")[-1].replace("db", ""))
        return add_noise_at_snr(clean, rng.choice(musan_music_files), snr)
    if condition == "reverb+musan_noise_5db":
        reverbed = add_reverb(clean, rng.choice(rir_files))
        return add_noise_at_snr(reverbed, rng.choice(musan_noise_files), 5.0)
    raise ValueError(f"Unknown condition: {condition}")

In [ ]:
# =====================================================================
# EMBEDDING + METRICS
# =====================================================================
def get_embedding(signal_tensor: torch.Tensor, model, device) -> torch.Tensor:
    signal = signal_tensor.unsqueeze(0).float().to(device)
    with torch.no_grad():
        emb = model.encode_batch(signal).squeeze(1)
        return F.normalize(emb, p=2, dim=-1)


def calculate_eer(pos_scores, neg_scores):
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return 0.0, 0.0
    pos, neg = np.array(pos_scores), np.array(neg_scores)
    thresholds = np.sort(np.concatenate([pos, neg]))
    min_diff, best_eer, best_thresh = float("inf"), 1.0, 0.0
    for t in thresholds:
        far = np.mean(neg >= t)
        frr = np.mean(pos < t)
        diff = abs(far - frr)
        if diff < min_diff:
            min_diff, best_eer, best_thresh = diff, (far + frr) / 2.0, t
    return best_eer * 100, best_thresh


def far_frr_at_threshold(pos_scores, neg_scores, threshold):
    pos, neg = np.array(pos_scores), np.array(neg_scores)
    far = float(np.mean(neg >= threshold)) * 100
    frr = float(np.mean(pos < threshold)) * 100
    return far, frr


CONDITIONS = [
    "clean", "reverb",
    "rirs_noise_15db", "rirs_noise_5db", "rirs_noise_0db",
    "musan_noise_15db", "musan_noise_5db", "musan_noise_0db",
    "musan_music_10db",
    "reverb+musan_noise_5db",
]


# =====================================================================
# PER-LANGUAGE EXPERIMENT RUNNER
# =====================================================================
def run_language_experiment(language_name, positive_pairs, negative_pairs, eval_files,
                             ecapa_model, device, rir_files, rirs_noise_files,
                             musan_noise_files, musan_music_files, seed):
    rng = random.Random(seed)
    print(f"\nLoading clean audio for all {language_name} eval files...")
    clean_audio = {key: load_mono_16k(key) for key in eval_files}

    results = {}
    clean_threshold = None

    for condition in CONDITIONS:
        print(f"\n--- [{language_name}] Condition: {condition} ---")
        cache = {}
        for idx, key in enumerate(eval_files):
            if idx % 200 == 0 or idx == len(eval_files) - 1:
                print(f"   -> {idx + 1}/{len(eval_files)}")
            try:
                augmented = apply_condition(clean_audio[key], condition, rir_files,
                                             rirs_noise_files, musan_noise_files,
                                             musan_music_files, rng)
                cache[key] = get_embedding(augmented, ecapa_model, device)
            except Exception as e:
                print(f"      [Warning] Skipping {key}: {e}")

        cos = torch.nn.CosineSimilarity(dim=-1)
        pos_scores = [cos(cache[a], cache[b]).item()
                      for a, b in positive_pairs if a in cache and b in cache]
        neg_scores = [cos(cache[a], cache[b]).item()
                      for a, b in negative_pairs if a in cache and b in cache]

        eer, thresh = calculate_eer(pos_scores, neg_scores)
        if condition == "clean":
            clean_threshold = thresh  # this language's own deployed operating point
        far_fixed, frr_fixed = far_frr_at_threshold(pos_scores, neg_scores, clean_threshold)

        results[condition] = {
            "language": language_name,
            "eer_retuned": eer,
            "far_at_clean_threshold": far_fixed,
            "frr_at_clean_threshold": frr_fixed,
            "n_pos": len(pos_scores), "n_neg": len(neg_scores),
        }
        print(f"   EER (re-tuned): {eer:.2f}%  |  FAR/FRR @ clean threshold: "
              f"{far_fixed:.2f}% / {frr_fixed:.2f}%")

    return results

In [ ]:
# =====================================================================
# MAIN
# =====================================================================
def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device.type.upper()}")

    print("\n=== Loading model ===")
    ecapa = EncoderClassifier.from_hparams(
        source="/content/drive/MyDrive/Data_Science_Project/Models/ECAPA-TDNN",
        savedir="tmpdir_ecapa", run_opts={"device": str(device)},
    )

    print("\n=== Discovering noise/RIR data ===")
    rir_files = discover_rir_files(args["rirs_root"])
    rirs_noise_files = discover_rirs_noise_files(args["rirs_root"])
    musan_noise_files = discover_musan_files(args["musan_root"], "noise")
    musan_music_files = discover_musan_files(args["musan_root"], "music")

    print("\n=== Discovering speech data ===")
    sinhala_speakers = discover_sinhala_speakers(args["sinhala_audio_dir"], args["sinhala_tsv"])
    english_speakers = discover_librispeech_speakers(args["english_dir"])

    print("\n=== Building trials ===")
    print("Sinhala:")
    sin_pos, sin_neg, sin_eval = build_balanced_trials(sinhala_speakers, args["utts_per_speaker"], args["seed"])
    print("English:")
    eng_pos, eng_neg, eng_eval = build_balanced_trials(english_speakers, args["utts_per_speaker"], args["seed"])

    all_results = {}
    all_results["Sinhala"] = run_language_experiment(
        "Sinhala", sin_pos, sin_neg, sin_eval, ecapa, device,
        rir_files, rirs_noise_files, musan_noise_files, musan_music_files, args["seed"],
    )
    all_results["English"] = run_language_experiment(
        "English", eng_pos, eng_neg, eng_eval, ecapa, device,
        rir_files, rirs_noise_files, musan_noise_files, musan_music_files, args["seed"],
    )

    print("\n" + "=" * 110)
    print("PHASE 3 -- NOISE/REVERB RESILIENCE, SINHALA + ENGLISH (ECAPA-TDNN, RIRS_NOISES + MUSAN)")
    print("=" * 110)
    header = f"{'Condition':<24}"
    for lang in all_results:
        header += f"{lang+' EER':>14}{lang+' FAR':>12}{lang+' FRR':>12}"
    print(header)
    for condition in CONDITIONS:
        row = f"{condition:<24}"
        for lang in all_results:
            r = all_results[lang][condition]
            row += f"{r['eer_retuned']:>13.2f}%{r['far_at_clean_threshold']:>11.2f}%{r['frr_at_clean_threshold']:>11.2f}%"
        print(row)
    print("=" * 110)

    rows = []
    for lang, cond_results in all_results.items():
        for cond, r in cond_results.items():
            rows.append({"condition": cond, **r})
    df = pd.DataFrame(rows)
    df.to_csv("phase3_noise_resilience_bilingual_results.csv", index=False)
    print("\nSaved: phase3_noise_resilience_bilingual_results.csv")
    return all_results


In [7]:
if __name__ == "__main__":
    args = {
        "sinhala_audio_dir": "/content/sinhala_audio",
        "sinhala_tsv": "/content/utt_spk_text.tsv",
        "english_dir": "/content/LibriSpeech/train-clean-100",
        "rirs_root": "/content/RIRS_NOISES",
        "musan_root": "/content/musan",
        "utts_per_speaker": 4,
        "seed": 42
    }

    main(args)

Device: CUDA

=== Loading model ===


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 83.3MB            

embedding_model.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


...

classifier.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


label_encoder.txt:   0%|          | 0.00/129k [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder



=== Discovering noise/RIR data ===
Found 60000 RIR files
Found 843 RIRS_NOISES point-source noise files
Found 930 MUSAN 'noise' files
Found 660 MUSAN 'music' files

=== Discovering speech data ===
[Sinhala] 478 speakers available locally
[English] 251 speakers available locally

=== Building trials ===
Sinhala:
   -> 478 speakers, 1912 files, 2868 positive / 2868 negative pairs
English:
   -> 251 speakers, 1004 files, 1506 positive / 1506 negative pairs

Loading clean audio for all Sinhala eval files...

--- [Sinhala] Condition: clean ---
   -> 1/1912
   -> 201/1912
   -> 401/1912
   -> 601/1912
   -> 801/1912
   -> 1001/1912
   -> 1201/1912
   -> 1401/1912
   -> 1601/1912
   -> 1801/1912
   -> 1912/1912
   EER (re-tuned): 4.46%  |  FAR/FRR @ clean threshold: 4.46% / 4.46%

--- [Sinhala] Condition: reverb ---
   -> 1/1912
   -> 201/1912
   -> 401/1912
   -> 601/1912
   -> 801/1912
   -> 1001/1912
   -> 1201/1912
   -> 1401/1912
   -> 1601/1912
   -> 1801/1912
   -> 1912/1912
   EER (r

## Phase 3 — Noise/Reverb Resilience, Sinhala + English (ECAPA-TDNN, RIRS_NOISES + MUSAN)

| Condition | Sinhala EER | Sinhala FAR | Sinhala FRR | English EER | English FAR | English FRR |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| clean | 4.46% | 4.46% | 4.46% | 0.33% | 0.33% | 0.33% |
| reverb | 6.17% | 4.15% | 7.81% | 0.46% | 0.40% | 0.60% |
| rirs_noise_15db | 5.26% | 4.15% | 6.38% | 0.33% | 0.27% | 0.33% |
| rirs_noise_5db | 6.76% | 3.73% | 11.37% | 0.80% | 0.33% | 1.39% |
| rirs_noise_0db | 8.86% | 4.08% | 18.62% | 1.39% | 0.53% | 2.26% |
| musan_noise_15db | 5.26% | 4.04% | 6.21% | 0.60% | 0.53% | 0.60% |
| musan_noise_5db | 7.43% | 3.73% | 11.89% | 0.73% | 0.46% | 0.80% |
| musan_noise_0db | 9.14% | 3.66% | 18.24% | 0.86% | 0.46% | 1.59% |
| musan_music_10db | 6.76% | 3.91% | 8.68% | 0.66% | 0.33% | 0.80% |
| reverb+musan_noise_5db | 10.01% | 3.63% | 21.86% | 1.46% | 0.40% | 2.86% |

In [8]:
"""
PHASE 3 -- Noise & reverb resilience characterization (Sinhala + English)
Uses RIRS_NOISES (SLR28) and MUSAN (noise, music, speech/babble) for noise diversity.
Self-contained.
"""

import argparse
import glob
import itertools
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio

SAMPLE_RATE = 16000

# Fix random seeds for full reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier


# =====================================================================
# DATA DISCOVERY
# =====================================================================
def discover_sinhala_speakers(audio_dir: str, tsv_path: str) -> dict:
    metadata = pd.read_csv(tsv_path, sep="\t", header=None,
                            names=["FileID", "UserID", "Transcription"])
    found_files = glob.glob(str(Path(audio_dir) / "**" / "*"), recursive=True)
    found_files = [f for f in found_files if f.lower().endswith((".wav", ".flac"))]
    stem_to_path = {Path(f).stem: f for f in found_files}
    metadata["path"] = metadata["FileID"].map(stem_to_path)
    available = metadata.dropna(subset=["path"])
    speaker_to_files = available.groupby("UserID")["path"].apply(list).to_dict()
    print(f"[Sinhala] {len(speaker_to_files)} speakers available locally")
    return speaker_to_files


def discover_librispeech_speakers(root_dir: str) -> dict:
    speaker_to_files = {}
    for speaker_dir in sorted(Path(root_dir).iterdir()):
        if not speaker_dir.is_dir():
            continue
        files = glob.glob(str(speaker_dir / "**" / "*.flac"), recursive=True)
        if files:
            speaker_to_files[speaker_dir.name] = files
    print(f"[English] {len(speaker_to_files)} speakers available locally")
    return speaker_to_files


def build_balanced_trials(speaker_to_files: dict, utts_per_speaker: int, seed: int = 42):
    rng = random.Random(seed)
    eligible = {s: files for s, files in speaker_to_files.items() if len(files) >= 2}
    capped = {s: files[:utts_per_speaker] for s, files in eligible.items()}

    positive_pairs = []
    for spk, files in capped.items():
        positive_pairs.extend(itertools.combinations(files, 2))
    rng.shuffle(positive_pairs)

    speakers = list(capped.keys())
    file_to_speaker = {f: s for s, files in capped.items() for f in files}
    negative_pairs = []
    for f1, _ in positive_pairs:
        spk1 = file_to_speaker[f1]
        other_spk = rng.choice([s for s in speakers if s != spk1])
        other_file = rng.choice(capped[other_spk])
        negative_pairs.append((f1, other_file))

    eval_files = list({f for pair in positive_pairs for f in pair} |
                       {f for pair in negative_pairs for f in pair})
    print(f"   -> {len(capped)} speakers, {len(eval_files)} files, "
          f"{len(positive_pairs)} positive / {len(negative_pairs)} negative pairs")
    return positive_pairs, negative_pairs, eval_files


# =====================================================================
# NOISE / RIR FILE DISCOVERY (three corpora subsets)
# =====================================================================
def discover_rir_files(rirs_root: str) -> list:
    files = glob.glob(str(Path(rirs_root) / "simulated_rirs" / "**" / "*.wav"), recursive=True)
    if not files:
        raise RuntimeError(f"No RIR files found under {rirs_root}/simulated_rirs")
    print(f"Found {len(files)} RIR files")
    return files


def discover_rirs_noise_files(rirs_root: str) -> list:
    files = glob.glob(str(Path(rirs_root) / "pointsource_noises" / "*.wav"))
    if not files:
        raise RuntimeError(f"No noise files found under {rirs_root}/pointsource_noises")
    print(f"Found {len(files)} RIRS_NOISES point-source noise files")
    return files


def discover_musan_files(musan_root: str, subset: str) -> list:
    files = glob.glob(str(Path(musan_root) / subset / "**" / "*.wav"), recursive=True)
    if not files:
        raise RuntimeError(f"No MUSAN '{subset}' files found under {musan_root}/{subset}")
    print(f"Found {len(files)} MUSAN '{subset}' files")
    return files


# =====================================================================
# AUDIO LOADING + AUGMENTATION
# =====================================================================
def load_mono_16k(path: str) -> torch.Tensor:
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    return waveform.squeeze(0)


def _rms(x: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.mean(x ** 2) + 1e-12)


def add_reverb(clean: torch.Tensor, rir_path: str) -> torch.Tensor:
    rir = load_mono_16k(rir_path)
    rir = rir / (torch.max(torch.abs(rir)) + 1e-8)
    augmented = torchaudio.functional.fftconvolve(clean, rir, mode="full")[: clean.shape[-1]]
    augmented = augmented.squeeze()  # Ensure 1D
    # Match original precision
    augmented = augmented.to(dtype=clean.dtype)
    augmented = augmented * (_rms(clean) / (_rms(augmented) + 1e-8))
    return augmented


def add_noise_at_snr(clean: torch.Tensor, noise_path: str, snr_db: float) -> torch.Tensor:
    noise = load_mono_16k(noise_path)
    if noise.shape[-1] < clean.shape[-1]:
        reps = int(np.ceil(clean.shape[-1] / noise.shape[-1]))
        noise = noise.repeat(reps)
    noise = noise[: clean.shape[-1]]
    clean_rms = _rms(clean)
    noise_rms = _rms(noise)
    target_noise_rms = clean_rms / (10 ** (snr_db / 20))
    noise = noise * (target_noise_rms / (noise_rms + 1e-8))
    return clean + noise


def apply_condition(clean: torch.Tensor, condition: str, rir_files: list,
                     rirs_noise_files: list, musan_noise_files: list,
                     musan_music_files: list, musan_speech_files: list,
                     rng: random.Random) -> torch.Tensor:
    if condition == "clean":
        return clean
    if condition == "reverb":
        return add_reverb(clean, rng.choice(rir_files))

    # RIRS point-source noises
    if condition.startswith("rirs_noise_"):
        snr = float(condition.split("_")[-1].replace("db", ""))
        return add_noise_at_snr(clean, rng.choice(rirs_noise_files), snr)

    # MUSAN noise (stationary)
    if condition.startswith("musan_noise_"):
        snr = float(condition.split("_")[-1].replace("db", ""))
        return add_noise_at_snr(clean, rng.choice(musan_noise_files), snr)

    # MUSAN music
    if condition.startswith("musan_music_"):
        snr = float(condition.split("_")[-1].replace("db", ""))
        return add_noise_at_snr(clean, rng.choice(musan_music_files), snr)

    # MUSAN speech (babble) – new addition
    if condition.startswith("musan_speech_"):
        snr = float(condition.split("_")[-1].replace("db", ""))
        return add_noise_at_snr(clean, rng.choice(musan_speech_files), snr)

    # Combined reverb + noise
    if condition == "reverb+musan_noise_5db":
        reverbed = add_reverb(clean, rng.choice(rir_files))
        return add_noise_at_snr(reverbed, rng.choice(musan_noise_files), 5.0)

    # Combined reverb + babble (optional, you can add more combos)
    if condition == "reverb+musan_speech_5db":
        reverbed = add_reverb(clean, rng.choice(rir_files))
        return add_noise_at_snr(reverbed, rng.choice(musan_speech_files), 5.0)

    raise ValueError(f"Unknown condition: {condition}")


# =====================================================================
# EMBEDDING + METRICS
# =====================================================================
def get_embedding(signal_tensor: torch.Tensor, model, device) -> torch.Tensor:
    signal = signal_tensor.unsqueeze(0).float().to(device)
    with torch.no_grad():
        emb = model.encode_batch(signal).squeeze(1)
        return F.normalize(emb, p=2, dim=-1)


def calculate_eer(pos_scores, neg_scores):
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return 0.0, 0.0
    pos, neg = np.array(pos_scores), np.array(neg_scores)
    thresholds = np.sort(np.concatenate([pos, neg]))
    min_diff, best_eer, best_thresh = float("inf"), 1.0, 0.0
    for t in thresholds:
        far = np.mean(neg >= t)
        frr = np.mean(pos < t)
        diff = abs(far - frr)
        if diff < min_diff:
            min_diff, best_eer, best_thresh = diff, (far + frr) / 2.0, t
    return best_eer * 100, best_thresh


def far_frr_at_threshold(pos_scores, neg_scores, threshold):
    pos, neg = np.array(pos_scores), np.array(neg_scores)
    far = float(np.mean(neg >= threshold)) * 100
    frr = float(np.mean(pos < threshold)) * 100
    return far, frr


# Updated conditions list with babble noise
CONDITIONS = [
    "clean", "reverb",
    "rirs_noise_15db", "rirs_noise_5db", "rirs_noise_0db",
    "musan_noise_15db", "musan_noise_5db", "musan_noise_0db",
    "musan_music_10db",
    "musan_speech_15db", "musan_speech_5db", "musan_speech_0db",
    "reverb+musan_noise_5db", "reverb+musan_speech_5db",
]


# =====================================================================
# PER-LANGUAGE EXPERIMENT RUNNER
# =====================================================================
def run_language_experiment(language_name, positive_pairs, negative_pairs, eval_files,
                             ecapa_model, device, rir_files, rirs_noise_files,
                             musan_noise_files, musan_music_files, musan_speech_files, seed):
    rng = random.Random(seed)
    print(f"\nLoading clean audio for all {language_name} eval files...")
    clean_audio = {key: load_mono_16k(key) for key in eval_files}

    results = {}
    clean_threshold = None

    for condition in CONDITIONS:
        print(f"\n--- [{language_name}] Condition: {condition} ---")
        cache = {}
        for idx, key in enumerate(eval_files):
            if idx % 200 == 0 or idx == len(eval_files) - 1:
                print(f"   -> {idx + 1}/{len(eval_files)}")
            try:
                augmented = apply_condition(clean_audio[key], condition, rir_files,
                                             rirs_noise_files, musan_noise_files,
                                             musan_music_files, musan_speech_files, rng)
                cache[key] = get_embedding(augmented, ecapa_model, device)
            except Exception as e:
                print(f"      [Warning] Skipping {key}: {e}")

        cos = torch.nn.CosineSimilarity(dim=-1)
        pos_scores = [cos(cache[a], cache[b]).item()
                      for a, b in positive_pairs if a in cache and b in cache]
        neg_scores = [cos(cache[a], cache[b]).item()
                      for a, b in negative_pairs if a in cache and b in cache]

        eer, thresh = calculate_eer(pos_scores, neg_scores)
        if condition == "clean":
            clean_threshold = thresh
        far_fixed, frr_fixed = far_frr_at_threshold(pos_scores, neg_scores, clean_threshold)

        results[condition] = {
            "language": language_name,
            "eer_retuned": eer,
            "far_at_clean_threshold": far_fixed,
            "frr_at_clean_threshold": frr_fixed,
            "n_pos": len(pos_scores), "n_neg": len(neg_scores),
        }
        print(f"   EER (re-tuned): {eer:.2f}%  |  FAR/FRR @ clean threshold: "
              f"{far_fixed:.2f}% / {frr_fixed:.2f}%")

    return results


# =====================================================================
# MAIN
# =====================================================================
def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device.type.upper()}")

    print("\n=== Loading model ===")
    ecapa = EncoderClassifier.from_hparams(
        source=args["ecapa_model_path"],
        savedir="tmpdir_ecapa", run_opts={"device": str(device)},
    )

    print("\n=== Discovering noise/RIR data ===")
    rir_files = discover_rir_files(args["rirs_root"])
    rirs_noise_files = discover_rirs_noise_files(args["rirs_root"])
    musan_noise_files = discover_musan_files(args["musan_root"], "noise")
    musan_music_files = discover_musan_files(args["musan_root"], "music")
    musan_speech_files = discover_musan_files(args["musan_root"], "speech")

    print("\n=== Discovering speech data ===")
    sinhala_speakers = discover_sinhala_speakers(args["sinhala_audio_dir"], args["sinhala_tsv"])
    english_speakers = discover_librispeech_speakers(args["english_dir"])

    print("\n=== Building trials ===")
    print("Sinhala:")
    sin_pos, sin_neg, sin_eval = build_balanced_trials(sinhala_speakers, args["utts_per_speaker"], args["seed"])
    print("English:")
    eng_pos, eng_neg, eng_eval = build_balanced_trials(english_speakers, args["utts_per_speaker"], args["seed"])

    all_results = {}
    all_results["Sinhala"] = run_language_experiment(
        "Sinhala", sin_pos, sin_neg, sin_eval, ecapa, device,
        rir_files, rirs_noise_files, musan_noise_files, musan_music_files,
        musan_speech_files, args["seed"],
    )
    all_results["English"] = run_language_experiment(
        "English", eng_pos, eng_neg, eng_eval, ecapa, device,
        rir_files, rirs_noise_files, musan_noise_files, musan_music_files,
        musan_speech_files, args["seed"],
    )

    print("\n" + "=" * 120)
    print("PHASE 3 -- NOISE/REVERB RESILIENCE, SINHALA + ENGLISH (ECAPA-TDNN, RIRS + MUSAN full)")
    print("=" * 120)
    header = f"{'Condition':<28}"
    for lang in all_results:
        header += f"{lang+' EER':>14}{lang+' FAR':>12}{lang+' FRR':>12}"
    print(header)
    for condition in CONDITIONS:
        row = f"{condition:<28}"
        for lang in all_results:
            r = all_results[lang][condition]
            row += f"{r['eer_retuned']:>13.2f}%{r['far_at_clean_threshold']:>11.2f}%{r['frr_at_clean_threshold']:>11.2f}%"
        print(row)
    print("=" * 120)

    rows = []
    for lang, cond_results in all_results.items():
        for cond, r in cond_results.items():
            rows.append({"condition": cond, **r})
    df = pd.DataFrame(rows)
    df.to_csv("phase3_noise_resilience_bilingual_results.csv", index=False)
    print("\nSaved: phase3_noise_resilience_bilingual_results.csv")
    return all_results


In [9]:
if __name__ == "__main__":
    args = {
        "sinhala_audio_dir": "/content/sinhala_audio",
        "sinhala_tsv": "/content/utt_spk_text.tsv",
        "english_dir": "/content/LibriSpeech/train-clean-100",
        "rirs_root": "/content/RIRS_NOISES",
        "musan_root": "/content/musan",
        "ecapa_model_path": "/content/drive/MyDrive/Data_Science_Project/Models/ECAPA-TDNN",
        "utts_per_speaker": 4,
        "seed": 42
    }

    main(args)

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/tmpdir_ecapa/hyperparams.yaml'


Device: CUDA

=== Loading model ===


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/tmpdir_ecapa/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/tmpdir_ecapa/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/tmpdir_ecapa/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/tmpdir_ecapa/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder



=== Discovering noise/RIR data ===
Found 60000 RIR files
Found 843 RIRS_NOISES point-source noise files
Found 930 MUSAN 'noise' files
Found 660 MUSAN 'music' files
Found 426 MUSAN 'speech' files

=== Discovering speech data ===
[Sinhala] 478 speakers available locally
[English] 251 speakers available locally

=== Building trials ===
Sinhala:
   -> 478 speakers, 1912 files, 2868 positive / 2868 negative pairs
English:
   -> 251 speakers, 1004 files, 1506 positive / 1506 negative pairs

Loading clean audio for all Sinhala eval files...

--- [Sinhala] Condition: clean ---
   -> 1/1912
   -> 201/1912
   -> 401/1912
   -> 601/1912
   -> 801/1912
   -> 1001/1912
   -> 1201/1912
   -> 1401/1912
   -> 1601/1912
   -> 1801/1912
   -> 1912/1912
   EER (re-tuned): 4.46%  |  FAR/FRR @ clean threshold: 4.46% / 4.46%

--- [Sinhala] Condition: reverb ---
   -> 1/1912
   -> 201/1912
   -> 401/1912
   -> 601/1912
   -> 801/1912
   -> 1001/1912
   -> 1201/1912
   -> 1401/1912
   -> 1601/1912
   -> 1801

## Phase 3 — Noise/Reverb Resilience, Sinhala + English (ECAPA-TDNN, RIRS + MUSAN full)

| Condition | Sinhala EER | Sinhala FAR | Sinhala FRR | English EER | English FAR | English FRR |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| clean | 4.46% | 4.46% | 4.46% | 0.33% | 0.33% | 0.33% |
| reverb | 6.17% | 4.15% | 7.81% | 0.46% | 0.40% | 0.60% |
| rirs_noise_15db | 5.26% | 4.15% | 6.38% | 0.33% | 0.27% | 0.33% |
| rirs_noise_5db | 6.76% | 3.73% | 11.37% | 0.80% | 0.33% | 1.39% |
| rirs_noise_0db | 8.86% | 4.08% | 18.62% | 1.39% | 0.53% | 2.26% |
| musan_noise_15db | 5.26% | 4.04% | 6.21% | 0.60% | 0.53% | 0.60% |
| musan_noise_5db | 7.43% | 3.73% | 11.89% | 0.73% | 0.46% | 0.80% |
| musan_noise_0db | 9.14% | 3.66% | 18.24% | 0.86% | 0.46% | 1.59% |
| musan_music_10db | 6.76% | 3.91% | 8.68% | 0.66% | 0.33% | 0.80% |
| musan_speech_15db | 16.46% | 2.68% | 32.22% | 1.20% | 0.27% | 1.59% |
| musan_speech_5db | 32.32% | 0.87% | 78.45% | 5.44% | 0.80% | 10.69% |
| musan_speech_0db | 40.41% | 0.73% | 93.10% | 15.60% | 1.26% | 34.06% |
| reverb+musan_noise_5db | 10.95% | 3.31% | 23.05% | 1.93% | 0.60% | 5.05% |
| reverb+musan_speech_5db | 41.04% | 0.66% | 91.77% | 14.87% | 1.06% | 33.80% |

In [3]:
from google.colab import runtime
runtime.unassign()
